# Plate with Blind Hole — Geometry & Mesh

Creates a rectangular plate with a cylindrical pocket (blind hole — does **not** go through).  
Uses the **OpenCASCADE** kernel in Gmsh for robust boolean operations.

| Parameter | Value |
|---|---|
| Plate | 100 × 80 × 20 mm |
| Hole radius | 12 mm |
| Hole depth | 14 mm (leaves 6 mm floor) |
| Hole position | centred on top face |

In [1]:
import gmsh
import meshio
import numpy as np
from pathlib import Path

OUT_DIR = Path("../output")
OUT_DIR.mkdir(exist_ok=True)

MSH_FILE  = str(OUT_DIR / "plate_blind_hole.msh")
XDMF_FILE = str(OUT_DIR / "plate_blind_hole.xdmf")

## 1 — Geometry (OCC kernel)

The blind hole is created by a **boolean cut**: subtract a shorter cylinder from the plate box.  
Because the cylinder is shorter than the plate, the bottom of the hole stays closed.

In [2]:
# ── Geometry parameters ──────────────────────────────────────────────────────
plate_lx, plate_ly, plate_lz = 1.0, 1.0, 1.0   # plate dimensions
hole_r    = 0.12                                     # hole radius
hole_depth = 0.5                                    # blind depth (< plate_lz)

# Hole centred on the top face (x, y) = (plate_lx/2, plate_ly/2)
hole_cx, hole_cy = plate_lx / 2, plate_ly / 2

# ── Gmsh init ────────────────────────────────────────────────────────────────
gmsh.initialize()
gmsh.model.add("plate_blind_hole")
occ = gmsh.model.occ

# Plate box: origin at (0,0,0), grows in +z
plate = occ.addBox(0, 0, 0, plate_lx, plate_ly, plate_lz)

# Cylinder for the pocket: starts at top face (z = plate_lz), drills downward
# addCylinder(x, y, z,  dx, dy, dz,  r)
# dz = -hole_depth → drills into the plate from the top
cylinder = occ.addCylinder(hole_cx, hole_cy, plate_lz,
                            0, 0, -hole_depth,
                            hole_r)

# Boolean cut: plate − cylinder → blind hole
# removeObject=True, removeTool=True deletes the originals
result, _ = occ.cut(
    [(3, plate)],
    [(3, cylinder)],
    removeObject=True,
    removeTool=True,
)

occ.synchronize()

vol_tag = result[0][1]
print(f"Volume tag after cut: {vol_tag}")
print(f"Surfaces: {[s[1] for s in gmsh.model.getEntities(2)]}")

Volume tag after cut: 1                                                                                                        
Surfaces: [7, 8, 9, 10, 11, 12, 13, 14]


## 2 — Physical groups

Tagging surfaces makes it easy to apply boundary conditions in FEniCS / FEniCSx later.

In [3]:
# Identify surfaces by their bounding-box centre
def surface_centre(tag):
    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(2, tag)
    return np.array([(xmin+xmax)/2, (ymin+ymax)/2, (zmin+zmax)/2])

surfaces = gmsh.model.getEntities(2)
bottom_tags, top_tags, side_tags, hole_tags = [], [], [], []

for _, tag in surfaces:
    c = surface_centre(tag)
    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(2, tag)

    if np.isclose(zmax, 0.0, atol=1e-6):                        # bottom face
        bottom_tags.append(tag)
    elif np.isclose(zmin, plate_lz, atol=1e-6):                 # top face (flat ring)
        top_tags.append(tag)
    elif np.isclose(zmin, plate_lz - hole_depth, atol=1e-6) \
         and c[2] > 0 and c[2] < plate_lz:                      # hole floor
        hole_tags.append(tag)
    elif (xmax - xmin) < 1e-6 or (ymax - ymin) < 1e-6:         # vertical side walls
        side_tags.append(tag)
    else:
        hole_tags.append(tag)                                    # cylindrical wall

# Register physical groups (tag 1-4 for easy BC assignment)
gmsh.model.addPhysicalGroup(2, bottom_tags, tag=1, name="bottom")
gmsh.model.addPhysicalGroup(2, top_tags,    tag=2, name="top")
gmsh.model.addPhysicalGroup(2, side_tags,   tag=3, name="sides")
gmsh.model.addPhysicalGroup(2, hole_tags,   tag=4, name="hole")
gmsh.model.addPhysicalGroup(3, [vol_tag],   tag=5, name="volume")

print("bottom:", bottom_tags)
print("top:   ", top_tags)
print("sides: ", side_tags)
print("hole:  ", hole_tags)

bottom: [13]
top:    [11]
sides:  [9, 10, 12, 14]
hole:   [7, 8]


## 3 — Mesh

Refine around the **hole edge** with a distance field so the curved pocket walls get smaller elements.

In [4]:
# ── Global size bounds ───────────────────────────────────────────────────────
lc_far  = 0.1   # coarse away from hole
lc_near = .05   # fine near hole

# Distance field from the hole surfaces (cylindrical wall + floor)
f_dist = gmsh.model.mesh.field.add("Distance")
gmsh.model.mesh.field.setNumbers(f_dist, "SurfacesList", hole_tags)

# Threshold: ramp from lc_near (at the hole wall) to lc_far (15 mm away)
f_thresh = gmsh.model.mesh.field.add("Threshold")
gmsh.model.mesh.field.setNumber(f_thresh, "InField",   f_dist)
gmsh.model.mesh.field.setNumber(f_thresh, "SizeMin",   lc_near)
gmsh.model.mesh.field.setNumber(f_thresh, "SizeMax",   lc_far)
gmsh.model.mesh.field.setNumber(f_thresh, "DistMin",   0.0)
gmsh.model.mesh.field.setNumber(f_thresh, "DistMax",   15.0)

gmsh.model.mesh.field.setAsBackgroundMesh(f_thresh)

# Disable Gmsh's automatic size from points so our field dominates
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints",        0)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature",     0)

# Generate 3-D tet mesh
gmsh.model.mesh.generate(3)
gmsh.model.mesh.optimize("Netgen")  # improve tet quality

gmsh.write(MSH_FILE)
print(f"Wrote {MSH_FILE}")

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 13 (Circle)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 20%] Meshing curve 15 (Circle)
Info    : [ 30%] Meshing curve 16 (Line)
Info    : [ 30%] Meshing curve 17 (Line)
Info    : [ 40%] Meshing curve 18 (Line)
Info    : [ 50%] Meshing curve 19 (Line)
Info    : [ 50%] Meshing curve 20 (Line)
Info    : [ 60%] Meshing curve 21 (Line)
Info    : [ 70%] Meshing curve 22 (Line)
Info    : [ 70%] Meshing curve 23 (Line)
Info    : [ 80%] Meshing curve 24 (Line)
Info    : [ 90%] Meshing curve 25 (Line)
Info    : [ 90%] Meshing curve 26 (Line)
Info    : [100%] Meshing curve 27 (Line)
Info    : Done meshing 1D (Wall 0.0206266s, CPU 0.02085s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 7 (Cylinder, Frontal-Delaunay)
Info    : [ 20%] Meshing surface 8 (Plane, Frontal-Delaunay)
Info    : [ 30%] Meshing surface 9 (Plane, Frontal-Delaunay)
Info    : [ 40%] Meshing surface 10 (Plane, Frontal-Delaunay)
Info    : [ 60%] Meshing su

## 4 — Convert to XDMF (FEniCS-compatible)

In [5]:
m = meshio.read(MSH_FILE)

# Volume mesh (tetra)
tetra_cells = m.cells_dict.get("tetra", np.empty((0, 4), dtype=int))
tetra_data  = m.cell_data_dict.get("gmsh:physical", {}).get("tetra", None)

meshio.write(
    XDMF_FILE,
    meshio.Mesh(
        points=m.points,
        cells=[("tetra", tetra_cells)],
        cell_data={"markers": [tetra_data]} if tetra_data is not None else {},
    ),
)

# Surface mesh (triangles) — boundary conditions
tri_cells = m.cells_dict.get("triangle", np.empty((0, 3), dtype=int))
tri_data  = m.cell_data_dict.get("gmsh:physical", {}).get("triangle", None)

if len(tri_cells) > 0:
    surf_file = XDMF_FILE.replace(".xdmf", "_surfaces.xdmf")
    meshio.write(
        surf_file,
        meshio.Mesh(
            points=m.points,
            cells=[("triangle", tri_cells)],
            cell_data={"markers": [tri_data]} if tri_data is not None else {},
        ),
    )
    print(f"Wrote {surf_file}")

print(f"Wrote {XDMF_FILE}")
print(f"  Nodes : {m.points.shape[0]}")
print(f"  Tetras: {len(tetra_cells)}")


Wrote ../output/plate_blind_hole_surfaces.xdmf
Wrote ../output/plate_blind_hole.xdmf
  Nodes : 6930
  Tetras: 32150


## 5 — Quick visual check (Gmsh GUI)

Uncomment to open the interactive Gmsh window (works in a terminal, not in JupyterLab).

In [6]:
# gmsh.fltk.run()   # ← uncomment for GUI
gmsh.finalize()
print("Done.")

Done.
